Download SPY Data

In [40]:
import yfinance as yf
import pandas as pd

BIG_SMA = 100
SMALL_SMA = 50
ACCOUNT_SIZE = 10000
PERCENTAGE_INVESTED = 1.0
STOPLOSS_PERCENTAGE = 0.02

df = yf.download("SPY", period="2y", interval="4h", multi_level_index=False)

df[f"SMA{SMALL_SMA}"] = df["Close"].rolling(SMALL_SMA).mean()
df[f"SMA{BIG_SMA}"] = df["Close"].rolling(BIG_SMA).mean()

df.head()

[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume,SMA50,SMA100
2024-06-06 09:30:00-04:00,534.140015,535.419983,533.240112,534.979980,15736137,NaN,NaN
2024-06-06 13:30:00-04:00,534.630005,534.669983,532.679993,534.140015,10345482,NaN,NaN
2024-06-07 09:30:00-04:00,536.170105,536.890015,532.534973,533.659973,19585360,NaN,NaN
2024-06-07 13:30:00-04:00,533.960022,536.409973,533.489990,536.179993,15531980,NaN,NaN
2024-06-10 09:30:00-04:00,534.309998,535.140015,532.570007,533.179993,13525169,NaN,NaN


In [41]:
df["Buy"] = (df[f"SMA{SMALL_SMA}"] > df[f"SMA{BIG_SMA}"]) & (df[f"SMA{SMALL_SMA}"].shift(1) <= df[f"SMA{BIG_SMA}"].shift(1))
df["Sell"] = (df[f"SMA{SMALL_SMA}"] < df[f"SMA{BIG_SMA}"]) & (df[f"SMA{SMALL_SMA}"].shift(1) >= df[f"SMA{BIG_SMA}"].shift(1))

first_sell_date = df[df["Sell"]].index.min()
last_buy_date = df[df["Buy"]].index.max()

df.loc[first_sell_date, "Sell"] = False 
df.loc[last_buy_date, "Buy"] = False 

df.loc[df["Buy"] | df["Sell"], ["Buy", "Sell", f"SMA{SMALL_SMA}", f"SMA{BIG_SMA}"]].head()

,Buy,Sell,SMA50,SMA100
2024-09-11 09:30:00-04:00,True,False,549.272793,549.219391
2025-02-13 09:30:00-05:00,True,False,599.076179,599.031903
2025-03-11 09:30:00-04:00,False,True,594.826332,595.159061
2025-05-16 13:30:00-04:00,True,False,554.827670,553.696643
2026-03-05 09:30:00-05:00,False,True,687.328394,687.765258


In [42]:
df["In Position"] = pd.NA
df.loc[df["Buy"], "In Position"] = True
df.loc[df["Sell"], "In Position"] = False
df["In Position"] = df["In Position"].ffill().fillna(False)
df.loc[df["In Position"], ["Buy", "Sell", "In Position", "Close"]].head()

,Buy,Sell,In Position,Close
2024-09-11 09:30:00-04:00,True,False,True,548.890015
2024-09-11 13:30:00-04:00,False,False,True,554.359985
2024-09-12 09:30:00-04:00,False,False,True,557.840027
2024-09-12 13:30:00-04:00,False,False,True,559.010010
2024-09-13 09:30:00-04:00,False,False,True,562.760010


In [44]:
df["Position Change"] = (~df["In Position"].shift(1).fillna(False) & df["Buy"]) | (df["In Position"].shift(1).fillna(False) & df["Sell"])

events = df.loc[df["Position Change"], ["Position Change", "Close", "Buy", "Sell"]]

entries = events.loc[df["Buy"], "Close"]
exits = events.loc[df["Sell"], "Close"]

df["Stop Loss"] = entries * (1 - STOPLOSS_PERCENTAGE)
df["Stop Loss"] = df["Stop Loss"].ffill()
    
# if df["In Position"] and df["Close"] < df["Stop Loss"]:

try:
    trades = pd.DataFrame({'Entry Date': entries.index, 'Entry Price': entries.values, 'Exit Date': exits.index, 'Exit Price': exits.values})
except ValueError:
    print("Error: The number of buy and sell signals do not match. Please check the data. Buy signals:", len(entries), "Sell signals:", len(exits))
    raise

trades["Return"] = (trades["Exit Price"] - trades["Entry Price"]) / trades["Entry Price"]

total_return = (1 + trades["Return"]).cumprod().iloc[-1] - 1

print(f"Total Return: {total_return * 100:.2f}%")
df

Total Return: 14.48%


,Close,High,Low,Open,Volume,SMA50,SMA100,Buy,Sell,In Position,Position Change,Stop Loss
2024-06-06 09:30:00-04:00,534.140015,535.419983,533.240112,534.979980,15736137,NaN,NaN,False,False,False,False,NaN
2024-06-06 13:30:00-04:00,534.630005,534.669983,532.679993,534.140015,10345482,NaN,NaN,False,False,False,False,NaN
2024-06-07 09:30:00-04:00,536.170105,536.890015,532.534973,533.659973,19585360,NaN,NaN,False,False,False,False,NaN
2024-06-07 13:30:00-04:00,533.960022,536.409973,533.489990,536.179993,15531980,NaN,NaN,False,False,False,False,NaN
2024-06-10 09:30:00-04:00,534.309998,535.140015,532.570007,533.179993,13525169,NaN,NaN,False,False,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-03 13:30:00-04:00,754.190002,756.049988,753.900024,754.632507,21958796,739.435206,709.604456,False,False,False,False,582.413988
2026-06-04 09:30:00-04:00,757.109985,757.179993,751.469971,752.099976,23602783,740.379207,710.641506,False,False,False,False,582.413988
2026-06-04 13:30:00-04:00,757.049988,758.309998,756.799988,757.109985,20038047,741.288606,711.680005,False,False,False,False,582.413988
2026-06-05 09:30:00-04:00,744.030029,752.820007,742.460022,752.309998,35197650,741.853007,712.540106,False,False,False,False,582.413988
